## Build Races Dimension
1. Read silver race table
2. Read silver circuits table
3. Join the data from races with circuits using circuit_id
4. Select the required columns:
    -  races.season
    -  races.round
    -  races.race_name
    -  races.race_date
    -  circuits.circuit_name
    -  circuits.locality
    -  circuits.country
5. Write the transformed data to gold dim_races table

Below changes are required to implement incremental Load Processing
1. Accept batch_id as a parameter to the notebook
2. Process data for only the batch_id being passes in (i.e.,  filter reading from bronze using the batch_id)
3. Add created_timestamp, updated_timestamp and batch_id to the gold table
4. Merge the processed data to the gold table
    - created_timestamp should be only populated at the time of inserting/creating the record. It should not be updated during the merge update

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.gold-helpers

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_races"

#### Step 1 & 2 - Read source tables
-     circuits
-     races

In [0]:
circuits_df = spark.table(f"{catalog_name}.{silver_schema}.circuits").filter(F.col("batch_id") == v_batch_id)
races_df = spark.table(f"{catalog_name}.{silver_schema}.races").filter(F.col("batch_id") == v_batch_id)

#### Step 3 & 4 -  Join races with circuits using circuits_id
Select the required columns:
1. races.season
2. races.round
3. races.race_name
4. races.race_date
5. circuits.circuit_name
6. circuits.locality
7. circuits.country

In [0]:
dim_races_df = (
    races_df
    .join(circuits_df, circuits_df.circuit_id == races_df.circuit_id, "inner")
    .select(
        races_df.season,
        races_df.round,
        races_df.race_name,
        races_df.race_date,
        circuits_df.circuit_id,
        circuits_df.circuit_name,
        circuits_df.locality,
        circuits_df.country
    )
)

#### Step 5 - Write the transformed data to gold dim_races table

In [0]:
merge_condition="tgt.season = src.season AND tgt.round = src.round"
columns_to_update=[
        "race_name",
        "race_date",
        "circuit_name",
        "locality",
        "country"
    ]
write_to_gold(
    dim_races_df,
    target_table,
    merge_condition,
    columns_to_update
)